# Training Pipeline — Dataset-Agnostic (Bank Marketing default)

This notebook builds, upserts, and runs the SageMaker **training** pipeline
(Seed → Preprocess → Train → Evaluate → Quality Gate → Register). It is written to
work with **any** dataset that is described in `src/config/dataset_schema.yaml`; the
**default** is the UCI **Bank Marketing** dataset (target `subscribed`).

Everything is driven off two sources of truth so you never edit the pipeline code:

| Concern | Source of truth | How to switch datasets |
|---------|-----------------|------------------------|
| Which CSV lands in S3 | `DATASET_LOADER` module below | point it at your own loader module that exposes `ensure_training_data_downloaded()` |
| Target column, features, id/timestamp, XGBoost objective | `src/config/dataset_schema.yaml` (read via `src.config.schema`) | edit the YAML to describe your columns |
| Account / region / bucket / Athena db | env vars (or `.env`) resolved by `src/config/config.py` | set the env cell below |

> The pipeline itself is already **schema-driven**: `pipeline.py` reads
> `schema.target_column()`, `schema.target_type()`, and the XGBoost objective from the
> YAML at definition time. No pipeline edits are needed to change datasets.

**Deployment is a separate step** — after this notebook registers an Approved model,
run `2_deployment.ipynb` (or `python main.py pipeline ...`) to stand up the endpoint.

## 1. Setup and Configuration

In [1]:
# Install the project (editable) so `src...` imports resolve inside the notebook kernel.
! uv pip install --system -e ../

Using Python 3.12.13 environment at: /opt/conda


⠙ sagemaker-automated-drift-and-trend-monitoring==0.1.0                         

⠙ mlflow==3.12.0                                                                

⠹ mlflow==3.12.0                                                                

⠸ mlflow==3.12.0                                                                

...

███████████████░░░░░ [23/29] sagemaker-core==2.16.0                             

████████████████░░░░ [24/29] litestar==2.24.0                                   

█████████████████░░░ [26/29] sagemaker-automated-drift-and-trend-monitoring==0.1

███████████████████░ [28/29] geventhttpclient==2.0.2                            

Installed 29 packages in 3.72s
 + appdirs==1.4.4
 + deprecation==2.1.0
 + dynaconf==3.3.2
 + evidently==0.7.21
 + faker==40.32.0
 + gevent==26.5.0
 + geventhttpclient==2.0.2
 + iterative-telemetry==0.0.10
 + litestar==2.24.0
 + litestar-htmx==0.5.0
 + msgspec==0.21.1
 + multipart==2.0.0
 - plotly==6.0.1 (from file:///home/conda/feedstock_root/build_artifacts/plotly_1742240435426/work)
 + plotly==5.24.1
 + polyfactory==3.3.0
 + prettytable==3.18.0
 - pynacl==1.6.2 (from file:///home/conda/feedstock_root/build_artifacts/pynacl_1767323877135/work)
 + pynacl==1.6.0
 + rich-click==1.9.8
 + sagemaker==3.16.0
 + sagemaker-automated-drift-and-trend-monitoring==0.1.0 (from file:///home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring)
 - sagemaker-core==2.13.1 (from file:///home/conda/feedstock_root/build_artifacts/sagemaker-core_1780638329688/work)
 + sagemaker-core==2.16.0
 - sagemaker-mlops==1.12.0 (from file:///home/conda/feedstock_root/build_artifact

### 1a. Choose the dataset loader

`DATASET_LOADER` names the module that downloads/transforms the dataset and uploads the
predictions CSV to S3. It must expose `ensure_training_data_downloaded(*, force=False)`.

- **Default:** `src.setup.download_dataset` → UCI **Bank Marketing** (target `subscribed`).
- **Bring your own:** write a module with the same `ensure_training_data_downloaded()`
  entry point and set `DATASET_LOADER` to its import path. (The existing fraud loader is
  `src.setup.download_kaggle_dataset`.)

The loader is **idempotent**: it skips the download when the predictions CSV already
exists in S3. Pass `force=True` to re-download/re-upload.

In [2]:
import os
import sys
import importlib
from pathlib import Path

# Make `src...` importable whether the kernel starts in notebooks/ or the repo root.
_project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

# ---- CRITICAL: Set env vars BEFORE any import that touches src.config.config ----
# The config module resolves constants at import time and caches them as module-level
# variables. If another import (e.g. the dataset loader) triggers config import before
# these are set, the wrong defaults get baked in permanently for this kernel session.
os.environ.setdefault('ATHENA_DATABASE', 'fraud_detection')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')
os.environ.setdefault('PROJECT_NAME', 'fraud-detection-monitoring')
os.environ.setdefault('DATA_S3_PREFIX', 'fraud-detection/')

# ---- Dataset selection knob -------------------------------------------------
# Default = Bank Marketing. To use a different dataset, point this at your own
# loader module (must expose `ensure_training_data_downloaded(*, force=False)`).
DATASET_LOADER = 'src.setup.download_dataset'   # Bank Marketing (default)
# DATASET_LOADER = 'src.setup.download_kaggle_dataset'  # example: fraud dataset
# -----------------------------------------------------------------------------

_loader = importlib.import_module(DATASET_LOADER)
print(f"Using dataset loader: {DATASET_LOADER}")

# Idempotent: skips if the predictions CSV is already in S3. Use force=True to
# re-download (e.g. after editing the transform).
result = _loader.ensure_training_data_downloaded()  # force=True to re-download
print(result)

2026-07-21 04:39:25,041 [INFO] Downloading UCI Bank Marketing dataset from https://archive.ics.uci.edu/static/public/222/bank+marketing.zip …


Using dataset loader: src.setup.download_dataset


2026-07-21 04:39:25,670 [INFO] Extracting bank-additional-full.csv from zip archive…


2026-07-21 04:39:25,809 [INFO] Transforming to project schema (41188 rows)…


2026-07-21 04:39:26,623 [INFO] Wrote /home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/data/bank_marketing_predictions_final.csv (5.3 MB, 41188 rows)


2026-07-21 04:39:26,633 [INFO] Uploading to s3://fraud-detection-monitoring-data-329430715989/fraud-detection/data/predictions/data.csv …


2026-07-21 04:39:26,791 [INFO] Uploading to s3://fraud-detection-monitoring-data-329430715989/fraud-detection/data/bank_marketing_predictions_final.csv …


2026-07-21 04:39:26,928 [INFO] ✓ Predictions CSV ready at s3://fraud-detection-monitoring-data-329430715989/fraud-detection/data/predictions/data.csv — pipeline seed step will load it into Athena.


{'downloaded': True, 'bucket': 'fraud-detection-monitoring-data-329430715989', 'predictions_key': 'fraud-detection/data/predictions/data.csv'}


In [3]:
# ---------------------------------------------------------------------------
# SWITCHING DATASETS — runnable steps (commented out on purpose)
# ---------------------------------------------------------------------------
# The default flow above (Bank Marketing) needs NONE of these. Run them ONLY
# when moving to a different dataset. Uncomment and execute in order, from the
# repo root, with the same env overrides as section 1b (already set once you
# run cell 1b, so the ! shell-outs below inherit them via the kernel env).
#
# STEP 0 — Describe your columns in src/config/dataset_schema.yaml
#          (id, timestamp, target, features, auxiliary, split). The pipeline
#          reads this file — no pipeline code changes. Then re-run section 1c
#          above to confirm the schema the pipeline will use.
#
# STEP 1 — Point DATASET_LOADER (cell 1a) at a module exposing
#          ensure_training_data_downloaded(); re-run that cell to (re)seed the
#          predictions CSV to s3://<DATA_S3_BUCKET>/<DATA_S3_PREFIX>data/predictions/data.csv
#
# STEP 2 — Rebuild the Athena tables to the new schema, then sanity-check:
#
# !python -m src.setup.download_dataset                       # (re)seed CSV -> S3
# !python -m src.setup.create_athena_tables --force-recreate  # rebuild tables to schema
# !python -m src.setup.validate_dataset_schema \
#     --csv-s3-uri s3://$DATA_S3_BUCKET/${DATA_S3_PREFIX}data/predictions/data.csv
# !python -m src.setup.create_athena_tables --verify-only     # confirm all tables exist
#
# After these succeed, continue with section 2 onward (create/upsert + run the
# pipeline). Training always resumes from the START of the pipeline; the
# SeedAthenaTrainingData step is idempotent, so re-running is safe and cheap.
print("Dataset-switching steps are documented above (commented out). "
      "No action needed for the default Bank Marketing dataset.")

Dataset-switching steps are documented above (commented out). No action needed for the default Bank Marketing dataset.


### 1b. Environment, AWS session, and MLflow

Configuration resolves as **env var → `config.yaml` → default**. The `config.yaml`
defaults ship for a `ml-monitoring` / `us-west-2` project, so if your deployed stack
uses different names, set the overrides below **before** importing `src.config.config`.

For the reference Bank Marketing deployment (ProjectName `fraud-detection-monitoring`,
region `us-east-1`) the values below are already correct — adjust to match your stack.

In [4]:
import os

# ---- Account / region / data-plane overrides --------------------------------
# Set these to match the CloudFormation stack you deployed. Leaving one unset
# falls back to config.yaml, then to a hardcoded default. These MUST be set
# before importing src.config.config (module-level paths are built at import).
os.environ['ATHENA_DATABASE'] = 'fraud_detection'
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'
os.environ['PROJECT_NAME'] = 'fraud-detection-monitoring'
os.environ['DATA_S3_PREFIX'] = 'fraud-detection/'
# DATA_S3_BUCKET is derived as ${PROJECT_NAME}-data-${AccountId} when unset.
# Set it explicitly only if your bucket name doesn't follow that convention:
# os.environ.setdefault('DATA_S3_BUCKET', 'fraud-detection-monitoring-data-329430715989')
# -----------------------------------------------------------------------------

import boto3
from sagemaker.core.helper.session_helper import Session
from dotenv import load_dotenv

# .env (written by the CFN lifecycle script) provides resolved outputs. Env vars
# set above win over .env because we do NOT override existing keys here.
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if 'notebooks' in str(notebook_dir) else notebook_dir
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(env_path, override=False)
    print(f"Loaded environment from: {env_path}")
else:
    print(f"No .env at {env_path} — relying on env vars / config.yaml defaults")

from src.utils.aws_utils import get_execution_role
from src.config.config import (
    MLFLOW_MODEL_NAME, DATA_S3_BUCKET, ATHENA_DATABASE, AWS_DEFAULT_REGION,
)

region = os.getenv('AWS_DEFAULT_REGION', 'us-east-1')
sagemaker_client = boto3.client('sagemaker', region_name=region)
s3_client = boto3.client('s3', region_name=region)

role = get_execution_role()
sagemaker_session = Session()
default_bucket = sagemaker_session.default_bucket()

print("\n=== Resolved configuration ===")
print(f"  Region:            {region}")
print(f"  Execution role:    {role}")
print(f"  Data bucket:       {DATA_S3_BUCKET}")
print(f"  SageMaker bucket:  {default_bucket}")
print(f"  Athena database:   {ATHENA_DATABASE}")
print(f"  Model pkg group:   {MLFLOW_MODEL_NAME}")

mlflow_uri = os.getenv('MLFLOW_TRACKING_URI', '')
print(f"  MLflow URI:        {mlflow_uri or '(not set — MLflow logging disabled)'}")

Loaded environment from: /home/sagemaker-user/sample-mlops-bestpractices/sagemaker-automated-drift-and-trend-monitoring/.env
Using SageMaker execution role from environment: arn:aws:iam::329430715989:role/fraud-detection-monitoring-SageMakerExecutionRole
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml



=== Resolved configuration ===
  Region:            us-east-1
  Execution role:    arn:aws:iam::329430715989:role/fraud-detection-monitoring-SageMakerExecutionRole
  Data bucket:       fraud-detection-monitoring-data-329430715989
  SageMaker bucket:  sagemaker-us-east-1-329430715989
  Athena database:   fraud_detection
  Model pkg group:   ml-model
  MLflow URI:        arn:aws:sagemaker:us-east-1:329430715989:mlflow-app/app-IEX3BTLYWBGJ


### 1c. Inspect the active dataset schema

This reads `src/config/dataset_schema.yaml` through `src.config.schema` and prints the
**target, features, id, and timestamp** the pipeline will use. Switching datasets =
editing that YAML; this cell always reflects whatever is active (no fraud/bank
assumptions hardcoded).

In [5]:
from src.config import schema

target = schema.target_column()
target_type = schema.target_type()
features = schema.feature_names()
id_col = schema.identifier_column()
ts_col = schema.timestamp_column()

print("=== Active dataset schema (dataset_schema.yaml) ===")
print(f"  Identifier column: {id_col}")
print(f"  Timestamp column:  {ts_col}")
print(f"  Target column:     {target}  (type: {target_type})")
print(f"  Feature count:     {len(features)}")
print(f"  Features:          {features}")
print()
print("XGBoost objective is auto-derived from target type at pipeline-definition")
print("time (boolean → binary:logistic, integer → multi:softprob, double →")
print("reg:squarederror), unless overridden in config.yaml (training.objective).")

=== Active dataset schema (dataset_schema.yaml) ===
  Identifier column: client_id
  Timestamp column:  contact_timestamp
  Target column:     subscribed  (type: boolean)
  Feature count:     20
  Features:          ['age', 'job', 'marital', 'education', 'credit_default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed']

XGBoost objective is auto-derived from target type at pipeline-definition
time (boolean → binary:logistic, integer → multi:softprob, double →
reg:squarederror), unless overridden in config.yaml (training.objective).


## 2. Pipeline configuration

In [12]:
# Pipeline name is generic so it fits any dataset. Rename per dataset if you want
# separate pipelines side-by-side (e.g. "bank-marketing-training-pipeline").
PIPELINE_NAME = "training-pipeline"
PIPELINE_DESCRIPTION = "Dataset-agnostic training pipeline (Bank Marketing default)"

# Per-execution parameter overrides. Leave AthenaTable at the schema default
# unless you seeded into a differently-named table.
PIPELINE_PARAMS = {
    'AthenaTable': 'training_data',
    'ModelApprovalStatus': 'Approved',   # auto-approve so the model is deploy-ready
    # 'MinRocAuc': '0.70',               # quality-gate override if needed
}

# Training-only run: register the model but do NOT deploy here. Deployment is
# handled separately by 2_deployment.ipynb.
INCLUDE_DEPLOYMENT = False

print(f"Pipeline:            {PIPELINE_NAME}")
print(f"Include deployment:  {INCLUDE_DEPLOYMENT}")
print(f"Parameters:          {PIPELINE_PARAMS}")

## 3. Create / Update the pipeline

Uses the `create_ml_training_pipeline` factory. `upsert_pipeline` creates the pipeline
if it doesn't exist or updates the definition if it does.

In [13]:
import importlib
import src.config.config
import src.train_pipeline.pipeline

# Force-reload config first (picks up the env var we set in cell 1b),
# then reload pipeline.py so its module-level `from src.config.config import ...`
# re-executes against the fresh config values.
importlib.reload(src.config.config)
importlib.reload(src.train_pipeline.pipeline)

from src.train_pipeline.pipeline import create_ml_training_pipeline
from src.config.config import ATHENA_DATABASE as _db_check

print(f"ATHENA_DATABASE resolved to: {_db_check}")
assert _db_check == 'fraud_detection', f"Expected 'fraud_detection', got '{_db_check}'"

pipeline_builder = create_ml_training_pipeline(
    pipeline_name=PIPELINE_NAME,
    region=region,
    role=role,
)

result = pipeline_builder.upsert_pipeline(
    description=PIPELINE_DESCRIPTION,
    include_deployment=INCLUDE_DEPLOYMENT,
    tags=[
        {'Key': 'Dataset', 'Value': schema.target_column()},
        {'Key': 'Notebook', 'Value': 'training_pipeline_bank_marketing'},
    ],
)
print(result)

## 4. Start pipeline execution

The **first** step (`SeedAthenaTrainingData`) loads the predictions CSV into the Athena
`training_data` table (idempotent), so training always resumes from the very start of
the pipeline — you do not seed Athena manually.

In [14]:
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
execution_name = f"{PIPELINE_NAME}-{timestamp}"

pipeline_parameters = [
    {'Name': key, 'Value': str(value)} for key, value in PIPELINE_PARAMS.items()
]

response = sagemaker_client.start_pipeline_execution(
    PipelineName=PIPELINE_NAME,
    PipelineExecutionDisplayName=execution_name,
    PipelineParameters=pipeline_parameters,
)
CURRENT_EXECUTION_ARN = response['PipelineExecutionArn']
print(f"Started execution: {execution_name}")
print(f"ARN: {CURRENT_EXECUTION_ARN}")

## 5. Monitor pipeline execution

Polls step statuses until the execution finishes. Expected flow (training-only):
`SeedAthenaTrainingData → PreprocessData → TrainModel → EvaluateModel → CheckModelQuality → RegisterModel`.

In [15]:
import time

def monitor_execution(execution_arn, poll_seconds=30):
    """Poll a pipeline execution and print step transitions until it ends."""
    terminal = {'Succeeded', 'Failed', 'Stopped'}
    while True:
        desc = sagemaker_client.describe_pipeline_execution(
            PipelineExecutionArn=execution_arn
        )
        status = desc['PipelineExecutionStatus']
        steps = sagemaker_client.list_pipeline_execution_steps(
            PipelineExecutionArn=execution_arn
        )['PipelineExecutionSteps']

        print(f"\n[{datetime.now():%H:%M:%S}] Execution status: {status}")
        for step in steps:
            print(f"  {step['StepName']:<28} {step['StepStatus']}")

        if status in terminal:
            print(f"\nExecution finished with status: {status}")
            return status
        time.sleep(poll_seconds)

monitor_execution(CURRENT_EXECUTION_ARN)

In [16]:
def get_actual_metrics(execution_arn):
    """Fetch the evaluation metrics emitted by the EvaluateModel step.

    Reads evaluation.json from the evaluation output prefix. The quality gate
    (CheckModelQuality) compares binary_classification_metrics.roc_auc.value
    against the MinRocAuc parameter (default 0.70).
    """
    import json
    steps = sagemaker_client.list_pipeline_execution_steps(
        PipelineExecutionArn=execution_arn
    )['PipelineExecutionSteps']
    eval_step = next((s for s in steps if s['StepName'] == 'EvaluateModel'), None)
    if not eval_step or eval_step['StepStatus'] != 'Succeeded':
        print("EvaluateModel step not completed yet.")
        return None

    eval_uri = f"s3://{sagemaker_session.default_bucket()}/fraud-detection/evaluation/evaluation.json"
    print(f"Reading metrics from: {eval_uri}")
    bucket, _, key = eval_uri[len('s3://'):].partition('/')
    try:
        body = s3_client.get_object(Bucket=bucket, Key=key)['Body'].read()
        metrics = json.loads(body)
        print(json.dumps(metrics, indent=2))
        return metrics
    except Exception as e:
        print(f"Could not read evaluation.json: {e}")
        return None

# get_actual_metrics(CURRENT_EXECUTION_ARN)

## 6. Utility functions

In [17]:
def stop_execution(execution_arn):
    """Stop a running pipeline execution."""
    sagemaker_client.stop_pipeline_execution(PipelineExecutionArn=execution_arn)
    print(f"Stop requested for: {execution_arn}")

def list_recent_executions(max_results=10):
    """List recent executions of this pipeline."""
    resp = sagemaker_client.list_pipeline_executions(
        PipelineName=PIPELINE_NAME, MaxResults=max_results
    )
    for e in resp['PipelineExecutionSummaries']:
        print(f"  {e.get('PipelineExecutionDisplayName','?'):<40} "
              f"{e['PipelineExecutionStatus']:<12} {e['StartTime']:%Y-%m-%d %H:%M}")

def latest_registered_model():
    """Show the most recent model package in the group (what deployment picks up)."""
    resp = sagemaker_client.list_model_packages(
        ModelPackageGroupName=MLFLOW_MODEL_NAME,
        SortBy='CreationTime', SortOrder='Descending', MaxResults=5,
    )
    for p in resp['ModelPackageSummaryList']:
        print(f"  v{p['ModelPackageVersion']:<3} {p['ModelApprovalStatus']:<12} "
              f"{p['CreationTime']:%Y-%m-%d %H:%M}  {p['ModelPackageArn']}")

# list_recent_executions()
# latest_registered_model()

## 7. Quick reference

### Switching datasets (no code changes needed)
1. **Describe your columns** in `src/config/dataset_schema.yaml` (id, timestamp, target,
   features, auxiliary, split). The pipeline reads this file — no pipeline code changes.
2. **Update preprocessing config** in `config.yaml` → `preprocessing` section:
   - `raw_input_csv`: path to your raw CSV in `data/raw/`
   - `column_rename_map`: source column names → schema names
   - `categorical_columns`: which columns to label-encode
   - `target_transform`: source column + positive value for boolean target
   - `identifier` / `timestamp` / `predictions`: auto-generation settings
3. **Update model config** in `config.yaml`:
   - `data.csv_training_data`: output filename
   - `training.xgboost_params`: tune for your class balance
   - `drift_generation.default_drift`: update feature names
4. **Place raw CSV** in `data/raw/` and run:

================================================================================
ATHENA TABLES SETUP (schema-driven)
================================================================================
Database:    fraud_detection
S3 Bucket:   fraud-detection-monitoring-data-329430715989
Region:      us-east-1
Feature cnt: 20 (from dataset_schema.yaml)
================================================================================

Then run this notebook top to bottom (or `python main.py pipeline create/start
--pipeline-name training-pipeline --wait`).

### From which step does training continue?
Always from the **start of the pipeline**. `SeedAthenaTrainingData` is idempotent (a few-
second no-op when the table is already populated), so re-running is safe and cheap.

### Prediction column configuration
Feature tables (`training_data`, `evaluation_data`) are schema-driven and fully dataset-
agnostic. If your inference handler emits differently-named prediction/score columns,
update `inference.prediction_column` / `inference.probability_column` in `config.yaml` —
downstream code (drift Lambda, dashboards, batch transform) reads these constants
everywhere, so a single edit propagates.
